<a href="https://colab.research.google.com/github/Aliviaaz/data-science-projects/blob/main/Geospatial_Dataset_Conversion_PipeLine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Documentation I referenced to make this notebook:
### Google Colab Notebooks
- "Snippets: Drive" = https://colab.research.google.com/notebooks/snippets/drive.ipynb#scrollTo=-f-hfkapsiPc
- Official Google Developer documentation "REST Resource: files" = https://developers.google.com/workspace/drive/api/reference/rest/v2/files

### Other helpful Google Colab Notebooks to look at in the future
- "Snippets: Accessing Files" = https://colab.research.google.com/notebooks/snippets/accessing_files.ipynb
- Loading and saving data from external sources: "External data: Local Files, Drive, Sheets, and Cloud Storage" = https://colab.research.google.com/notebooks/io.ipynb
- "Snippets: Saving Data to Google Cloud Storage" = https://colab.research.google.com/notebooks/snippets/gcs.ipynb
- "Authenticate to GCP" = https://colab.research.google.com/notebooks/snippets/gcp_auth.ipynb#scrollTo=BIlYtG4CrIhx


In [39]:
# Import PyDrive and associated libraries.
# This only needs to be done once per notebook.
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

In [40]:
# Authenticate and create the PyDrive client.
# This only needs to be done once per notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

Search query reference:
- https://developers.google.com/drive/v2/web/search-parameters
- https://developers.google.com/workspace/drive/api/guides/search-shareddrives

PyDrive documentation:
- https://docs.iterative.ai/PyDrive2/filelist/#get-all-files-which-matches-the-query

How to find the driveID
- find the URL of the drive: https://drive.google.com/drive/folders/0ALFxMY3LZxmeUk9PVA
- the part after /folders/ is the id

In [41]:
# search in the Common Data drive for shape files
driveId = "0ALFxMY3LZxmeUk9PVA"
fileType = ".shp"

# store the results of the search in a list
file_list = drive.ListFile({
    'q': f"title contains '{fileType}'",
    'corpora': 'drive',
    'driveId': driveId,
    'includeItemsFromAllDrives': True
}).GetList()

The Google Drive API query language doesn't support endsWith or regex. This is why adding post-filter with Python's endswith() is necessary. Otherwise outputs could include files like "hydro_SHP_22-05-13.zip" which contain SHP in the title but are not actually shapefiles.

In [42]:
# Post-filter to only include files strictly ending in .shp
file_list = [f for f in file_list if f['title'].endswith('.shp')]

In [43]:
# print the results to show that it worked, and the
for file1 in file_list:
    print('title: {}, id: {}'.format(file1['title'], file1['id']))

title: All_Parcels.shp, id: 1j0NBBdlVO2eZBSr79qnUXThuYPfd-TtD
title: OR State Boundaries.shp, id: 1csBWDFfoKaR7Iu3HFEUr5okhB9j0d5rN
title: WesternCascadesCounties.shp, id: 1UfsrtnESn8fGdCowBHmQoi7q4yUnd88I
title: King_County_Parcels__parcel_area.shp, id: 1hbIjiZS_ZMpsbgEYB93MRhcHabzkdlhF
title: 20' Contours.shp, id: 1vYkER1dHNKcg1bRSx6fsfLvx6WMtgEV7
title: soilmu_a_wa641_joined.shp, id: 1pGT22w_WP2sP6ADUNRw1716khcV_O4ZH
title: parcels_NoOwner.shp, id: 1d3YXHxYQw4g-RvI5lpt9RtmOYbXe9ZdP
title: soilmu_p_wa641.shp, id: 1iRc8-W4oHvjx8zGxMz_JxCfxIjRmaNe-
title: soilsf_l_wa641.shp, id: 1kAsIE_tuG2DdHEvTz8wc2A8te91mPgro
title: soilmu_a_wa641.shp, id: 1hr7nBB1c8YErn4-SFnsEo_Xyzs6NDF08
title: soilsf_p_wa641.shp, id: 1jw9Xd-9K_3tC5a_F5eFDF590k5jSPEjU
title: soilmu_l_wa641.shp, id: 1iekpj_m65zoiXwdRmAiF4U5pT2CMOz63
title: soilsa_a_wa641.shp, id: 1i8RSSuVQpeiEho39hg066iLPKjIoPXKJ
title: soilsf_p_or640.shp, id: 1Z_g9MBeNJg2BVvxIWF5NNyh2eK0yJdL5
title: soilmu_p_or640.shp, id: 1YVsJ_PvPlkueo1oD42aLGjI

Documentation for .get() function
- https://docs.python.org/3/library/stdtypes.html#dict.get
- https://developers.google.com/workspace/drive/api/reference/rest/v2/files

In [44]:
# show each file's size in MB and KB
sorted_fileSize_asc = sorted(file_list, key=lambda x: int(x.get('fileSize', 0)))
index = 0
for file1 in sorted_fileSize_asc:
    size_bytes = int(file1.get('fileSize', 0))
    if size_bytes >= 1_000_000:
        size_str = f"{size_bytes / 1_000_000:.2f} MB"
    else:
        size_str = f"{size_bytes / 1_000:.2f} KB"
    print('index: ', index, 'title: {}, size: {}'.format(file1['title'], size_str))
    index += 1

index:  0 title: soilmu_p_wa641.shp, size: 0.10 KB
index:  1 title: soilmu_l_wa641.shp, size: 0.10 KB
index:  2 title: soilmu_p_or640.shp, size: 0.10 KB
index:  3 title: soilmu_l_or640.shp, size: 0.10 KB
index:  4 title: soilsf_l_or640.shp, size: 0.10 KB
index:  5 title: soilmu_p_wa633.shp, size: 0.10 KB
index:  6 title: soilmu_l_wa633.shp, size: 0.10 KB
index:  7 title: soilmu_l_wa749.shp, size: 0.10 KB
index:  8 title: soilmu_p_wa749.shp, size: 0.10 KB
index:  9 title: soilmu_p_wa651.shp, size: 0.10 KB
index:  10 title: soilmu_l_wa651.shp, size: 0.10 KB
index:  11 title: soilmu_p_wa649.shp, size: 0.10 KB
index:  12 title: soilmu_l_wa649.shp, size: 0.10 KB
index:  13 title: soilmu_l_wa648.shp, size: 0.10 KB
index:  14 title: soilmu_p_wa648.shp, size: 0.10 KB
index:  15 title: soilmu_l_wa619.shp, size: 0.10 KB
index:  16 title: soilmu_p_wa619.shp, size: 0.10 KB
index:  17 title: soilmu_l_wa608.shp, size: 0.10 KB
index:  18 title: soilmu_p_wa608.shp, size: 0.10 KB
index:  19 title: soil

There are a total of 17 files over 100 MB.
The next step is to convert these larger files to GeoParquet and upload them to the Google Storage Bucket.

In [45]:
from google.cloud import storage
import geopandas as gpd
import os
import shutil
import tempfile

Definition of function that creates a pipeline for converting larger vector datasets to GeoParquet.

In [46]:
REQUIRED_EXTENSIONS = ['.shx', '.dbf']
OPTIONAL_EXTENSIONS = ['.prj', '.cpg', '.sbn', '.sbx']
ALL_EXTENSIONS = REQUIRED_EXTENSIONS + OPTIONAL_EXTENSIONS


def get_all_companion_files(base_name, drive):
    return drive.ListFile({
        'q': f"title contains '{base_name}'",
        'corpora': 'drive',
        'driveId': driveId,
        'includeItemsFromAllDrives': True
    }).GetList()

Account for edge cases.

In [47]:
def get_companion_files(base_name, all_files_list):
    # Check for duplicate base names across different folders
    shp_files = [f for f in all_files_list if f['title'] == f"{base_name}.shp"]
    if len(shp_files) > 1:
        parent_ids = [f['parents'][0]['id'] for f in shp_files]
        raise ValueError(
            f"Duplicate base name '{base_name}' found in multiple folders: {parent_ids}. "
            f"Resolve ambiguity before proceeding."
        )

    companions = [
        f for f in all_files_list
        if any(f['title'] == f"{base_name}{ext}" for ext in ['.shp'] + ALL_EXTENSIONS)
    ]

    # Check required companions are present
    present_titles = [f['title'] for f in companions]
    missing = [ext for ext in REQUIRED_EXTENSIONS if f"{base_name}{ext}" not in present_titles]
    if missing:
        raise FileNotFoundError(
            f"Missing required companion files for '{base_name}': {missing}"
        )

    print(f"Found {len(companions)} companion files: {present_titles}")
    return companions


def check_disk_space(required_bytes, local_temp_dir="/tmp", multiplier=3):
    """Ensure enough disk space exists (multiplier accounts for input + output + buffer)."""
    free_bytes = shutil.disk_usage(local_temp_dir).free
    required = required_bytes * multiplier
    if free_bytes < required:
        raise OSError(
            f"Insufficient disk space. Required: {required / 1e6:.2f} MB, "
            f"Available: {free_bytes / 1e6:.2f} MB"
        )


def validate_geodataframe(gdf, base_name):
    # Check for empty file
    if len(gdf) == 0:
        raise ValueError(f"'{base_name}.shp' contains no features.")

    # Check for missing CRS
    if gdf.crs is None:
        raise ValueError(f"'{base_name}.shp' has no CRS defined.")

    # Check for null geometries
    null_count = gdf.geometry.isna().sum()
    if null_count > 0:
        print(f"Warning: {null_count} null geometries found in '{base_name}.shp'.")

    # Check for mixed geometry types
    geom_types = gdf.geometry.geom_type.unique()
    if len(geom_types) > 1:
        print(f"Warning: Mixed geometry types found in '{base_name}.shp': {geom_types}.")


def verify_upload(blob, local_output):
    """Verify uploaded file size matches local file size."""
    blob.reload()
    local_size = os.path.getsize(local_output)
    if blob.size != local_size:
        blob.delete()
        raise ValueError(
            f"Upload verification failed. Local size: {local_size}, "
            f"GCS size: {blob.size}. Blob deleted."
        )

In [35]:
def convert_and_upload_single_to_gcs(file1, drive, bucket_name, local_temp_dir="/tmp", gcs_folder="vector-data", timeout=300):

    client = storage.Client()
    bucket = client.bucket(bucket_name)

    title = file1['title']
    base_name = title.replace('.shp', '')
    size_bytes = int(file1.get('fileSize', 0))
    size_mb = size_bytes / 1_000_000
    output_filename = f"{base_name}.geoparquet"

    # Check if valid complete file already exists in GCS
    blob = bucket.blob(f"{gcs_folder}/{output_filename}")
    if blob.exists():
        blob.reload()
        if blob.size and blob.size > 0:
            print(f"Skipping {output_filename} — already exists in gs://{bucket_name}/{gcs_folder}/")
            return
        else:
            print(f"Incomplete upload detected for {output_filename} — re-uploading...")
            blob.delete()

    print(f"\nProcessing {title} ({size_mb:.2f} MB)...")

    # Check disk space before downloading
    check_disk_space(size_bytes, local_temp_dir)

    # Find companion files, raises on duplicates or missing required files
    all_files_list = get_all_companion_files(base_name, drive)
    companion_files = get_companion_files(base_name, all_files_list)

    local_input = os.path.join(local_temp_dir, title)
    local_output = os.path.join(local_temp_dir, output_filename)
    downloaded_files = []

    try:
        # Download companion files, track each individually for partial download cleanup
        for companion in companion_files:
            local_companion = os.path.join(local_temp_dir, companion['title'])
            try:
                companion.GetContentFile(local_companion)
                downloaded_files.append(local_companion)
            except Exception as e:
                raise IOError(f"Failed to download '{companion['title']}': {e}")

        gdf = gpd.read_file(local_input)

        # Validate data quality
        validate_geodataframe(gdf, base_name)

        print(f"Bounds: {gdf.total_bounds}")
        print(f"Features: {len(gdf)}")
        print(f"CRS: {gdf.crs}")

        gdf.to_parquet(local_output)

        # Upload with timeout for large files
        blob.upload_from_filename(local_output, timeout=timeout)

        # Verify upload completeness
        verify_upload(blob, local_output)

        print(f"Uploaded {output_filename} to gs://{bucket_name}/{gcs_folder}/{output_filename}")

    except Exception as e:
        print(f"Error processing {title}: {e}")

    finally:
        for f in downloaded_files:
            if os.path.exists(f):
                os.remove(f)
        if os.path.exists(local_output):
            os.remove(local_output)

In [48]:
def convert_and_upload_file_list_to_gcs(file_list, drive, bucket_name, min_size_mb=100, local_temp_dir="/tmp", gcs_folder="vector-data"):

    filtered_files = [
        f for f in file_list
        if f['title'].endswith('.shp') and int(f.get('fileSize', 0)) >= min_size_mb * 1_000_000
    ]

    print(f"Found {len(filtered_files)} .shp files above {min_size_mb} MB threshold")

    for file1 in filtered_files:
        convert_and_upload_single_to_gcs(file1, drive, bucket_name, local_temp_dir, gcs_folder)

In [49]:
# Safe indexed access
def get_file_by_index(sorted_list, index):
    if index >= len(sorted_list):
        raise IndexError(f"Index {index} out of range — file list has {len(sorted_list)} files.")
    return sorted_list[index]

In [51]:
# Test single file that already exists
convert_and_upload_single_to_gcs(
    file1=get_file_by_index(sorted_fileSize_asc, 228),
    drive=drive,
    bucket_name="cloud-gdb-raster-data"
)

Skipping WBDHU12.geoparquet — already exists in gs://cloud-gdb-raster-data/vector-data/


In [52]:
# Test single file that doesn't exist yet
convert_and_upload_single_to_gcs(
    file1=get_file_by_index(sorted_fileSize_asc, 227),
    drive=drive,
    bucket_name="cloud-gdb-raster-data"
)


Processing S_USA.RangerDistrict.shp (67.71 MB)...
Found 7 companion files: ['S_USA.RangerDistrict.sbn', 'S_USA.RangerDistrict.sbx', 'S_USA.RangerDistrict.shp', 'S_USA.RangerDistrict.cpg', 'S_USA.RangerDistrict.shx', 'S_USA.RangerDistrict.dbf', 'S_USA.RangerDistrict.prj']
Bounds: [-150.00792811   18.03374359  -65.69967      61.51899025]
Features: 503
CRS: EPSG:4269
Uploaded S_USA.RangerDistrict.geoparquet to gs://cloud-gdb-raster-data/vector-data/S_USA.RangerDistrict.geoparquet


"Warning: Mixed geometry types found in 'S_USA.RangerDistrict.shp': ['MultiPolygon' 'Polygon']." means that within the same shapefile, some features have Polygon geometry and others have MultiPolygon geometry.
- Polygon — a single continuous shape
- MultiPolygon — multiple disconnected shapes grouped as one feature
Although files can still be uploaded with mixed types, it might cause issues later if some spatial libraries or tools expect a single geometry type and may throw errors or behave unexpectedly.
A fix could be standardize everything to MultiPolygon before writing to parquet:

from shapely.geometry import MultiPolygon

gdf['geometry'] = gdf['geometry'].apply(
    lambda geom: MultiPolygon([geom]) if geom.geom_type == 'Polygon' else geom
)

Calling the function.

In [ ]:
convert_and_upload_file_list_to_gcs(
    file_list=file_list,
    bucket_name="cloud-gdb-raster-data",
    min_size_mb=100,
    gcs_folder="vector-data"
)